In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from datetime import timedelta


In [65]:
def prepare_data(df, asset_class=None):
    """
    Prepare the DataFrame for the investment window analysis by ensuring proper data types and sorting.
    
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')

    Returns:
        DataFrame: Cleaned and sorted DataFrame
    """
    # Create a copy to avoid modifying original data
    data = df.copy()
    
    # Filter by asset class if specified
    if asset_class is not None:
        data = data[data['TransactionClass'] == asset_class].copy()
        if data.empty:
            raise ValueError(f"No data found for asset class: {asset_class}")
    
    # Ensure TransactionDate is datetime
    data['TransactionDate'] = pd.to_datetime(data['TransactionDate'])
    
    # Sort by date to ensure chronological order
    data = data.sort_values('TransactionDate').reset_index(drop=True)
    
    # Use Available column as Balance for clarity
    # data['Balance'] = data['Available']
    data['TransactionClass'] = asset_class if asset_class else 'Not Specified'

    # Test print to verify
    # print(data[['TransactionDate', 'Balance','TransactionClass']])

    return data[['TransactionDate', 'Available','TransactionClass']]

In [66]:
def find_start_date_and_global_minimum(df):
    """
    Identify the start date and global minimum balance information.
    
    Args:
        df: DataFrame with 'TransactionDate' and 'Availble' columns
    
    Returns:
        tuple: (start_date, global_min_balance, global_min_dates)
    """
    start_date = df['TransactionDate'].min()
    global_min_balance = df['Available'].min()
    global_min_dates = df[df['Available'] == global_min_balance]['TransactionDate'].tolist()
    
    return start_date, global_min_balance, global_min_dates




In [67]:
def check_interval_validity(df_interval):
    """
    Check if an interval is valid according to the criteria:
    - The lowest balance in the interval occurs at the end date
    - If same low balance occurs on consecutive days, only capture the last occurrence
    
    Args:
        df_interval: DataFrame subset for the current interval
    
    Returns:
        tuple: (is_valid, min_balance, end_date)
    """
    if df_interval.empty:
        return False, None, None
    
    min_balance = df_interval['Available'].min()
    end_date = df_interval['TransactionDate'].iloc[-1]
    end_balance = df_interval['Available'].iloc[-1]
    
    # Check if the minimum balance occurs at the end date
    if end_balance != min_balance:
        return False, min_balance, end_date
    
    # Find all dates with the minimum balance
    min_balance_dates = df_interval[df_interval['Available'] == min_balance]['TransactionDate'].tolist()
    
    # If minimum balance occurs only at the end, it's valid
    if len(min_balance_dates) == 1 and min_balance_dates[0] == end_date:
        return True, min_balance, end_date
    
    # If minimum balance occurs on multiple days, check if end_date is the last occurrence
    if end_date == max(min_balance_dates):
        return True, min_balance, end_date
    
    return False, min_balance, end_date


In [68]:
def find_lowest_balance_intervals(df, asset_class=None):
    """
    Find all valid intervals where new lowest balances are set.
    
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
    
    Returns:
        list: List of dictionaries containing interval information
    """
    # Prepare data
    data = prepare_data(df, asset_class)
    
    # Print filtering information
    if asset_class:
        print(f"Filtering data for asset class: {asset_class}")
        print(f"Records found: {len(data)}")
        print()
    
    # Get basic information
    start_date, global_min_balance, global_min_dates = find_start_date_and_global_minimum(data)
    
    print(f"Start date: {start_date.date()}")
    print(f"Global minimum balance: ${global_min_balance:,.2f}")
    print(f"Global minimum occurs on: {[d.date() for d in global_min_dates]}")
    print()
    
    valid_intervals = []
    current_lowest_balance = float('inf')
    
    # Get unique dates for iteration
    unique_dates = sorted(data['TransactionDate'].unique())
    start_idx = 0  # Index of start_date in unique_dates
    
    # Iterate through progressively larger intervals
    for end_idx in range(start_idx, len(unique_dates)):
        current_start_date = unique_dates[start_idx]
        current_end_date = unique_dates[end_idx]
        
        # Filter data for current interval
        mask = (data['TransactionDate'] >= current_start_date) & (data['TransactionDate'] <= current_end_date)
        df_interval = data[mask].copy()
        
        # Check if interval is valid
        is_valid, min_balance, end_date = check_interval_validity(df_interval)
        
        if is_valid and min_balance < current_lowest_balance:
            # This is a new lowest balance
            interval_info = {
                'start_date': current_start_date,
                'end_date': current_end_date,
                'balance': min_balance,
                'days_in_interval': (current_end_date - current_start_date).days + 1
            }
            
            valid_intervals.append(interval_info)
            current_lowest_balance = min_balance
            
            print(f"Valid interval found: {current_start_date.date()} to {current_end_date.date()}")
            print(f"  New lowest balance: ${min_balance:,.2f}")
            print(f"  Interval length: {interval_info['days_in_interval']} days")
            print()
    
    return valid_intervals

In [69]:
def create_results_dataframe(intervals):
    """
    Convert the list of intervals to a DataFrame for easy viewing.
    
    Args:
        intervals: List of interval dictionaries
    
    Returns:
        DataFrame: Results in tabular format
    """
    if not intervals:
        return pd.DataFrame(columns=['Start_Date', 'End_Date', 'Available', 'Days_in_Interval'])
    
    results_df = pd.DataFrame(intervals)
    results_df = results_df.rename(columns={
        'start_date': 'Start_Date',
        'end_date': 'End_Date',
        'balance': 'Available',
        'days_in_interval': 'Days_in_Interval'
    })
    
    return results_df

In [70]:
def analyze_balance_data(df, asset_class=None):
    """
    Main function to perform the complete balance analysis.
    
    Args:
        df: DataFrame with 'TransactionDate', 'TransactionClass', and 'Available' columns
        asset_class: String to filter TransactionClass column (e.g., 'US Agencies')
    
    Returns:
        DataFrame: Results showing valid intervals where new lowest balances are set
    """
    print("=== Balance Analysis ===")
    print()
    
    # Find valid intervals
    intervals = find_lowest_balance_intervals(df, asset_class)
    
    # Create results DataFrame
    results_df = create_results_dataframe(intervals)
    
    print("=== Final Results ===")
    if results_df.empty:
        print("No valid intervals found.")
    else:
        print(results_df.to_string(index=False))
    
    return results_df

In [71]:
# # Example usage:
# if __name__ == "__main__":
    
#     df = pd.DataFrame(sample_data)
    
#     # Run the analysis for all data
#     print("Analysis for all asset classes:")
#     results_all = analyze_balance_data(df)
    
#     print("\n" + "="*50 + "\n")
    
#     # Run the analysis filtered by asset class
#     print("Analysis filtered for 'US Agencies':")
#     results_filtered = analyze_balance_data(df, asset_class='US Agencies')

In [ ]:
# DEBUG TEST CELL for running the asset class balances processing

asset_classes = ['Certificate of Deposit', 'Mutual Fund', 'Commercial Paper', 'Money Market', 'US Treasuries', 'US Agencies']

# Load the data
running_balances = pd.read_pickle('running_balances.pkl')
data = prepare_data(running_balances, asset_classes[2])
# print(asset_classes[2])
# print(prepare_data(running_balances, asset_classes[2]))

# Base information
# start_date, global_min_balance, global_min_dates = find_start_date_and_global_minimum(data)

# print(f"Start date: {start_date.date()}")
# print(f"Global minimum balance: ${global_min_balance:,.2f}")
# print(f"Global minimum occurs on: {[d.date() for d in global_min_dates]}")
# print()

# valid_intervals = []
# current_lowest_balance = float('inf') # Initialize with infinity

# Get unique dates for iteration
# unique_dates = sorted(data['TransactionDate'].unique())
# start_idx = 0  # Index of start_date in unique_dates

# print(unique_dates[0])
# print(unique_dates)

# Find valid intervals
# intervals = find_lowest_balance_intervals(data, asset_classes[2])

# Pretty print the intervals with formatted values
# for i, interval in enumerate(intervals, 1):
#     print(f"\nInterval {i}:")
#     print(f"  Start Date: {interval['start_date'].strftime('%Y-%m-%d')}")
#     print(f"  End Date: {interval['end_date'].strftime('%Y-%m-%d')}")
#     print(f"  Balance: ${interval['balance']:,.2f}")
#     print(f"  Days: {interval['days_in_interval']}")


Filtering data for asset class: Commercial Paper
Records found: 396

Start date: 2025-09-04
Global minimum balance: $0.00
Global minimum occurs on: [datetime.date(2025, 9, 10), datetime.date(2025, 9, 25), datetime.date(2025, 9, 26), datetime.date(2025, 9, 27), datetime.date(2025, 9, 28), datetime.date(2025, 9, 29), datetime.date(2025, 9, 30), datetime.date(2025, 10, 1), datetime.date(2025, 10, 2), datetime.date(2025, 10, 3), datetime.date(2025, 10, 4), datetime.date(2025, 10, 5), datetime.date(2025, 10, 6), datetime.date(2025, 10, 7), datetime.date(2025, 10, 8), datetime.date(2025, 10, 9), datetime.date(2025, 10, 10), datetime.date(2025, 10, 11), datetime.date(2025, 10, 12), datetime.date(2025, 10, 24), datetime.date(2025, 10, 25), datetime.date(2025, 10, 26), datetime.date(2025, 10, 27), datetime.date(2025, 10, 28), datetime.date(2025, 10, 29), datetime.date(2025, 10, 30), datetime.date(2025, 10, 31), datetime.date(2025, 11, 1), datetime.date(2025, 11, 2), datetime.date(2025, 11, 3), 